In [7]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# =========================
# Config
# =========================
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE   = "../data/splits/shared_split.json"

DL_MODEL_FILE    = "../models/bilstm_model.keras"
TOKENIZER_FILE   = "../models/tokenizer.json"
LABEL_MAP_FILE   = "../models/label_mapping.json"
DL_METADATA_FILE = "../models/dl_metadata.json"
DL_PRED_FILE     = "../data/processed/dl_test_predictions.csv"
DL_SUMMARY_FILE  = "../data/processed/dl_summary.csv"

MAX_VOCAB_SIZE = 10000
MAX_LEN = 50
EMBED_DIM = 128
LSTM_UNITS = 64
BATCH_SIZE = 16
EPOCHS = 10

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# =========================
# Load data
# =========================
df = pd.read_csv(SHARED_DATASET_FILE)

with open(SHARED_SPLIT_FILE, "r", encoding="utf-8") as f:
    split_dict = json.load(f)

train_df = df[df["row_id"].isin(split_dict["train_ids"])].copy()
val_df   = df[df["row_id"].isin(split_dict["val_ids"])].copy()
test_df  = df[df["row_id"].isin(split_dict["test_ids"])].copy()

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# =========================
# Tokenizer
# =========================
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(train_df["text"])

X_train = tokenizer.texts_to_sequences(train_df["text"])
X_val   = tokenizer.texts_to_sequences(val_df["text"])
X_test  = tokenizer.texts_to_sequences(test_df["text"])

X_train_pad = pad_sequences(X_train, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad  = pad_sequences(X_test, maxlen=MAX_LEN, padding="post", truncating="post")

# =========================
# Labels
# =========================
label_encoder = LabelEncoder()

y_train = label_encoder.fit_transform(train_df["label"])
y_val   = label_encoder.transform(val_df["label"])
y_test  = label_encoder.transform(test_df["label"])

num_classes = len(label_encoder.classes_)

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_val_cat   = to_categorical(y_val, num_classes=num_classes)
y_test_cat  = to_categorical(y_test, num_classes=num_classes)

print("Classes:", label_encoder.classes_)

# =========================
# Model
# =========================
vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)

model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(LSTM_UNITS)),
    Dropout(0.5),
    Dense(num_classes, activation="softmax")
])

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

# =========================
# Train
# =========================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

model.fit(
    X_train_pad,
    y_train_cat,
    validation_data=(X_val_pad, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

# =========================
# Evaluate
# =========================
test_pred_prob = model.predict(X_test_pad, verbose=0)
test_pred_classes = np.argmax(test_pred_prob, axis=1)
test_pred_labels = label_encoder.inverse_transform(test_pred_classes)
test_true_labels = label_encoder.inverse_transform(y_test)

acc = accuracy_score(test_true_labels, test_pred_labels)
f1_macro = f1_score(test_true_labels, test_pred_labels, average="macro")
f1_weighted = f1_score(test_true_labels, test_pred_labels, average="weighted")

print("\n===== DL RESULT =====")
print("Accuracy    :", acc)
print("F1 Macro    :", f1_macro)
print("F1 Weighted :", f1_weighted)
print("\nClassification Report:")
print(classification_report(test_true_labels, test_pred_labels))
print("Confusion Matrix:")
print(confusion_matrix(test_true_labels, test_pred_labels))

# =========================
# Save
# =========================
model.save(DL_MODEL_FILE)

with open(TOKENIZER_FILE, "w", encoding="utf-8") as f:
    f.write(tokenizer.to_json())

label_mapping = {label: int(i) for i, label in enumerate(label_encoder.classes_)}
with open(LABEL_MAP_FILE, "w", encoding="utf-8") as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=2)

metadata = {
    "max_sequence_length": MAX_LEN,
    "classes": label_encoder.classes_.tolist(),
    "max_vocab_size": MAX_VOCAB_SIZE
}
with open(DL_METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

pred_df = test_df[["row_id", "text", "label"]].copy()
pred_df["dl_pred"] = test_pred_labels
pred_df["dl_confidence"] = test_pred_prob.max(axis=1)
pred_df.to_csv(DL_PRED_FILE, index=False, encoding="utf-8-sig")

summary_df = pd.DataFrame([{
    "Model": "BiLSTM",
    "Accuracy": acc,
    "F1_Macro": f1_macro,
    "F1_Weighted": f1_weighted
}])

summary_df.to_csv(DL_SUMMARY_FILE, index=False, encoding="utf-8-sig")

print("\n===== Summary =====")
print(summary_df)

print("\nSaved:")
print("-", DL_MODEL_FILE)
print("-", TOKENIZER_FILE)
print("-", LABEL_MAP_FILE)
print("-", DL_METADATA_FILE)
print("-", DL_PRED_FILE)
print("-", DL_SUMMARY_FILE)
print("Done")

Train: (1512, 3)
Val  : (324, 3)
Test : (324, 3)
Classes: ['angry' 'disgust' 'fear' 'happy' 'neutral' 'sad']


c:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.3664 - loss: 1.5572 - val_accuracy: 0.5802 - val_loss: 1.2442
Epoch 2/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.7546 - loss: 0.7926 - val_accuracy: 0.8549 - val_loss: 0.4643
Epoch 3/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9120 - loss: 0.2860 - val_accuracy: 0.8951 - val_loss: 0.2877
Epoch 4/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9451 - loss: 0.1654 - val_accuracy: 0.9259 - val_loss: 0.2173
Epoch 5/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9709 - loss: 0.1044 - val_accuracy: 0.8580 - val_loss: 0.3364
Epoch 6/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 31ms/step - accuracy: 0.9775 - loss: 0.0675 - val_accuracy: 0.9228 - val_loss: 0.2626
Epoch 7/10
95/95 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.9934 - loss: 0.0343 - val_accuracy: 0.9105 - val_loss: 0.3017

===== DL RESULT =====
Accuracy    : 0.9012345679012346
F1 Macro    : 0.9132275733292298
F1 Weighted : 0.898391